**Azure Databricks — Hands-on Assessment**

**Scenario: Smart Parking Management System**

**Part 1 — Dataset preparation**

**Step 1 — Create the source Delta table**

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
    session_id INT,
    zone_id STRING,
    vehicle_type STRING,
    duration_hours INT,
    parking_fee INT,
    status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


**Step 2 — Create a working Delta table**

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


**Step 3 — Create the zone master table**

In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
    zone_id STRING,
    zone_name STRING,
    city STRING,
    capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


**Step 4 — Prepare incoming records for MERGE**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
);

**Part 2 — Assessment questions**

**Section A — SQL and DataFrames**

1. Display all records from parking_sessions_source . Retrieve only session_id , vehicle_type , and parking_fee.

In [0]:
%sql
SELECT * FROM parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


In [0]:
%sql
SELECT session_id, vehicle_type, parking_fee 
FROM parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


2. Find all completed parking sessions where the fee is greater than ₹100.

In [0]:
%sql
SELECT * FROM parking_sessions_source
WHERE status = 'Completed' AND parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


3. Calculate the total parking fees and number of completed sessions for each vehicle type using GROUP BY.

In [0]:
%sql
SELECT vehicle_type,
        SUM(parking_fee) AS total_parking_fees,
        COUNT(*) AS completed_sessions
FROM parking_sessions_source
WHERE status = 'Completed'
GROUP BY vehicle_type
ORDER BY vehicle_type;

vehicle_type,total_parking_fees,completed_sessions
Bike,160,4
Car,1750,8
SUV,540,2


4. Using PySpark, read parking_sessions_source into a DataFrame, add a column named fee_category ( High for fees of ₹200 or more, otherwise Low ), and display the result.

In [0]:
from pyspark.sql.functions import col, when
 
df = spark.table("parking_sessions_source")
df = df.withColumn("fee_category", when(col("parking_fee") >= 200, "High").otherwise("Low"))
display(df)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


**Section B — Delta Tables and CRUD** 

5. Create a new managed Delta table named parking_summary using CTAS, containing only completed sessions.


In [0]:
%sql
CREATE OR REPLACE TABLE parking_summary
USING DELTA
AS
SELECT * FROM parking_sessions_source WHERE status = 'Completed';

SELECT * FROM parking_summary;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed


6. Insert a new record into parking_sessions_work with session ID 123, zone Z01, vehicle type Car, duration 3 hours, fee ₹150, and status Active.

In [0]:
%sql
INSERT INTO parking_sessions_work VALUES (123, 'Z01', 'Car', 3, 150, 'Active');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT * FROM parking_sessions_work WHERE session_id = 123;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
123,Z01,Car,3,150,Active


7. Update session 104 to Completed and increase its parking fee to ₹280.

In [0]:
%sql
UPDATE parking_sessions_work
SET status = 'Completed', parking_fee = 280
WHERE session_id = 104;

num_affected_rows
1


8. Delete all Cancelled sessions from parking_sessions_work . Display the remaining records.

In [0]:
%sql
DELETE FROM parking_sessions_work WHERE status = 'Cancelled';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,280,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


**Section C — MERGE and Delta Operations**

9. Display incoming_parking_sessions and identify which records already exist in parking_sessions_work.

In [0]:
%sql
SELECT * FROM incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
SELECT i.*,
CASE WHEN w.session_id IS NOT NULL THEN 'Exists (will UPDATE)'
ELSE 'New (will INSERT)' END AS merge_action
FROM incoming_parking_sessions i
LEFT JOIN parking_sessions_work w
ON i.session_id = w.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,merge_action
102,Z02,Bike,3,60,Completed,Exists (will UPDATE)
104,Z01,SUV,5,300,Completed,Exists (will UPDATE)
121,Z03,Car,4,200,Completed,New (will INSERT)
122,Z05,Bike,2,40,Active,New (will INSERT)


10. Perform a MERGE operation using session_id to update existing records and insert new records.

In [0]:
%sql
MERGE INTO parking_sessions_work AS t
USING incoming_parking_sessions AS s
ON t.session_id = s.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
SELECT * FROM parking_sessions_work
WHERE session_id IN (102, 104, 121, 122) ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


11. Use DESCRIBE HISTORY to display the transaction history of parking_sessions_work.

In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-11-06T04:41:04.000Z,72240446752486,tsaravanapriya535@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1017261170416928),ea523f91-26e7-42f7-9f12-185ea6d84122,1106-042209-crmkty3h-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9846, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-06T04:41:03.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(session_id#14299 = session_id#14281)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1017261170416928),ea523f91-26e7-42f7-9f12-185ea6d84122,1106-042209-crmkty3h-v2n,7,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7568, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3463, materializeSourceTimeMs -> 306, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1112, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1971)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-06T04:37:01.000Z,72240446752486,tsaravanapriya535@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1017261170416928),7b83db91-8ad1-4cc9-929f-55215791a8ea,1106-042209-crmkty3h-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-06T04:36:59.000Z,72240446752486,tsaravanapriya535@gmail.com,DELETE,"Map(predicate -> [""(status#13655 = Cancelled)""])",null,List(1017261170416928),7b83db91-8ad1-4cc9-929f-55215791a8ea,1106-042209-crmkty3h-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 963, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 494, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 467)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T04:36:32.000Z,72240446752486,tsaravanapriya535@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1017261170416928),04324eab-5615-48fe-933e-698ad2ddde0a,1106-042209-crmkty3h-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1878)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:36:18.000Z,72240446752486,tsaravanapriya535@gmail.com,DELETE,"Map(predicate -> [""(session_id#13142 = 123)""])",null,List(1017261170416928),738f5d87-180f-4257-8643-1661f2fabf54,1106-042209-crmkty3h-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3756, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 787, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 630, numAddedFiles 

12. Use Time Travel to retrieve the table version immediately before the MERGE operation. Compare session 102 before and after MERGE.

In [0]:
%sql
SELECT max(version) AS version_before_merge
FROM (DESCRIBE HISTORY parking_sessions_work)
WHERE version < (
  SELECT version FROM (DESCRIBE HISTORY parking_sessions_work)
  WHERE operation = 'MERGE'
);

version_before_merge
7


In [0]:
%sql
SELECT * FROM parking_sessions_work VERSION AS OF 7
ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,280,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
SELECT 'Before MERGE' AS state, * FROM parking_sessions_work VERSION AS OF 7 WHERE session_id = 102
UNION ALL
SELECT 'Current' AS state, * FROM parking_sessions_work WHERE session_id = 102;

state,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
Before MERGE,102,Z02,Bike,2,40,Completed
Current,102,Z02,Bike,3,60,Completed


**Section D — Tables, Views and Optimization**

13. Create a permanent view named completed_parking_view containing only completed sessions from parking_sessions_work.

In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT * FROM parking_sessions_work WHERE status = 'Completed';

SELECT * FROM completed_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


14. Create a temporary view containing parking sessions from zone Z01 and a global temporary view containing sessions with parking fees above ₹200. Query both views.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW z01_sessions_tmp AS
SELECT * FROM parking_sessions_work WHERE zone_id = 'Z01';

SELECT * FROM z01_sessions_tmp;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_sessions_global AS
SELECT * FROM parking_sessions_work WHERE parking_fee > 200;

SELECT * FROM global_temp.high_fee_sessions_global;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7521064467550936>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_sessions_global AS\nSELECT * FROM parking_sessions_work WHERE parking_fee > 200;\n\nSELECT * FROM global_temp.high_fee_sessions_global;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbrun

15. Join parking_sessions_work with parking_zones and display the zone name, city, vehicle type, and parking fee.

In [0]:
%sql
SELECT z.zone_name, z.city, w.vehicle_type, w.parking_fee
FROM parking_sessions_work w
JOIN parking_zones z ON w.zone_id = z.zone_id
ORDER BY z.zone_name;

zone_name,city,vehicle_type,parking_fee
Airport Parking,Delhi,Car,250
Airport Parking,Delhi,Bike,40
Airport Parking,Delhi,SUV,300
Airport Parking,Delhi,Bike,20
Central Plaza,Hyderabad,Bike,80
Central Plaza,Hyderabad,SUV,180
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,SUV,300
City Mall,Mumbai,Car,350


16. Run DESCRIBE DETAIL on parking_sessions_work , execute OPTIMIZE , and compare the file count before and after optimization. Explain why a small dataset might show no significant change.

In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,87656ace-f725-4ded-87f1-d91d20cade5e,workspace.default.parking_sessions_work,null,,2026-11-06T04:23:32.816Z,2026-11-06T04:41:04.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
OPTIMIZE parking_sessions_work;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793940852885, 1793940853816, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,87656ace-f725-4ded-87f1-d91d20cade5e,workspace.default.parking_sessions_work,null,,2026-11-06T04:23:32.816Z,2026-11-06T04:41:04.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


**Why a small dataset may show no significant change:**
OPTIMIZE compacts many small files into fewer larger ones (target ~1 GB each). This table holds only a few dozen rows(a few KB), and the CREATE/INSERT/UPDATE/DELETE/MERGE operations produced only a handful of small files. Even after compaction the data still fits in one file, so numFiles may stay the same or drop only slightly. OPTIMIZE pays off with large tables that have accumulated many small files (for example from frequent small writes or streaming).

**Section E — Structured Streaming and DLT**

17. Using PySpark's built-in rate source, generate five streaming records per second and display the stream.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.checkpoints;

In [0]:
rate_df = (spark.readStream
           .format("rate")
           .option("rowsPerSecond", 5)
           .load())

display(rate_df, checkpointLocation="/Volumes/workspace/default/checkpoints/q17_rate")

Checkpointing to /Volumes/workspace/default/checkpoints/q17_rate


18. Apply a streaming transformation that creates an amount column from value and filters records with an amount greater than ₹300.

In [0]:
from pyspark.sql.functions import col

amount_df = (spark.readStream
             .format("rate")
             .option("rowsPerSecond", 5)
             .load()
             .withColumn("amount", col("value") * 10)
             .filter(col("amount") > 300))

display(amount_df, checkpointLocation="/Volumes/workspace/default/checkpoints/q18_amount")

Checkpointing to /Volumes/workspace/default/checkpoints/q18_amount


19. Create a Lakeflow/DLT SQL pipeline with a Bronze streaming table reading from parking_sessions_source and a Silver streaming table retaining only completed sessions.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE parking_bronze
COMMENT 'Raw parking sessions streamed from parking_sessions_source'
AS SELECT * FROM STREAM(parking_sessions_source);

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE parking_silver
COMMENT 'Completed parking sessions only'
AS SELECT *
   FROM STREAM(parking_bronze)
   WHERE status = 'Completed';

message
The operation was successfully executed.


20. Create a Gold materialized view that calculates total completed sessions and total parking revenue by zone. Run the pipeline and verify the Gold output.

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW parking_gold
COMMENT 'Completed sessions and revenue by zone'
AS SELECT zone_id,
          COUNT(*) AS total_completed_sessions,
          SUM(parking_fee) AS total_parking_revenue
   FROM parking_silver
   GROUP BY zone_id;

result
The operation was successfully executed.
